# 评价阶段整理：用注释看懂每一步的粒度

本次从原始 CSV 独立复现：完整明细 → 每单最新评价 → 订单聚合 → 全部订单关联。

请先保存第八份 Notebook，再在这里选择内核逐格运行。函数位于 src/reviews_cleaning.py；关键筛选、聚合代码也在下面展开解释。

预期：完整评价 99,224 行，订单级评价 98,673 行，全部订单关联 99,441 行。不同粒度的行数本来就可以不同，我们还要通过控制总额检查是否遗漏记录。

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# 这是项目根目录。以后换电脑，只需先修改这里。
project_dir = Path(r'D:\数据分析\olist-operations-analytics')
data_dir = project_dir.parent / 'archive'

# 让当前 Notebook 能导入 src 文件夹中的公共函数。
src_dir = str(project_dir / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from reviews_cleaning import (
    clean_order_reviews, select_latest_reviews, aggregate_order_reviews,
    join_orders_reviews, reconcile_review_totals, compare_review_rules,
    check_review_keys, check_review_text, check_review_dates,
    summarize_review_metrics
)

def read_raw(filename):
    # ID 和日期先按字符串读取，随后再进行有检查的类型转换。
    # 不把字面上的 NA 等文本自动识别为空；这里只将空字段识别为缺失。
    return pd.read_csv(
        data_dir / filename,
        dtype=str,
        encoding='utf-8-sig',
        keep_default_na=False,
        na_values=['']
    )

# 两张原表一直保留，下面所有清洗和聚合都返回新表。
raw_reviews = read_raw('olist_order_reviews_dataset.csv')
raw_orders = read_raw('olist_orders_dataset.csv')
print('原始评价记录数：', len(raw_reviews))
print('原始订单数：', len(raw_orders))

原始评价记录数： 99224
原始订单数： 99441


## 1. 清洗后的明细仍然是一行一条评价

clean_order_reviews 返回工作副本：检查组合键，转换评分、日期，生成文字存在标记和时间标记。没有标题或正文的有效评分仍保留。

低评分定义是评分 ≤ 2，即 1 或 2 分。原始评价 ID、订单 ID 的单列重复不等于组合键重复。

In [2]:
# 此函数不会把同一订单的多条评价合并；输出仍是评价记录粒度。
reviews_clean = clean_order_reviews(raw_reviews)

display(check_review_keys(raw_reviews))
display(check_review_text(raw_reviews))
display(check_review_dates(raw_reviews, reviews_clean))

print('清洗前后评价记录数：', len(raw_reviews), len(reviews_clean))
print('低评分评价记录数：', reviews_clean['is_low_score'].sum())
reviews_clean['review_score'].value_counts().sort_index()

评价记录行数           99224
评价ID不同值数         98410
涉及订单数            98673
评价ID缺失行数             0
订单ID缺失行数             0
评价ID重复行数_首条之后      814
订单ID重复行数_首条之后      551
组合主键重复行数_首条之后        0
整行重复数_首条之后           0
Name: 数量, dtype: int64

,原始缺失数,非空但只有空白字符数,无有效文字记录数,有有效文字记录数
review_comment_title,87656,2,87658,11566
review_comment_message,58247,27,58274,40950


,原始缺失数,非空但解析失败数,转换后类型
review_creation_date,0,0,datetime64[ns]
review_answer_timestamp,0,0,datetime64[ns]


清洗前后评价记录数： 99224 99224
低评分评价记录数： 14575


review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: Int64

## 2. 在哪里筛选“每单最新一条”

select_latest_reviews 内部的关键代码是：

```python
# transform 将同一订单的最大提交时间对应到每一行，尚未减少行数。
latest_time = cleaned.groupby("order_id")[
    "review_answer_timestamp"
].transform("max")

# loc 才真正筛选：只选提交时间等于该订单最大时间的记录。
latest = cleaned.loc[
    cleaned["review_answer_timestamp"].eq(latest_time)
].copy()
```

当前最新时间没有并列，所以筛选后每单一行。若新数据存在最新时间并列、提交时间缺失或时间倒置，函数先报错核查；不会任意取原始表最后一行。

In [3]:
# 生成一张新表，保留每单最新的一条评价；reviews_clean 仍有全部明细。
latest_reviews = select_latest_reviews(reviews_clean)

pd.Series({
    '完整评价工作表行数': len(reviews_clean),
    '每单最新评价表行数': len(latest_reviews),
    '最新评价表的重复订单数': latest_reviews['order_id'].duplicated().sum()
})

完整评价工作表行数      99224
每单最新评价表行数      98673
最新评价表的重复订单数        0
dtype: int64

## 3. 在哪里把全部评价汇总为每单一行

aggregate_order_reviews 内部的聚合是：

```python
stats = cleaned.groupby("order_id", as_index=False).agg(
    review_record_count=("review_id", "size"),       # 该订单所有评价记录数
    low_score_record_count=("is_low_score", "sum"),  # 该订单低评分记录数
    any_low_score=("is_low_score", "any"),          # 是否至少有一条低评分
)
```

size、sum、any 计算的是不同信息。然后将这张统计表与 latest_reviews 连接，订单级结果同时保存“全部历史记录的信息”和“最新评价的信息”。

validate='one_to_one' 检查两张输入表是否每单一行，本身不负责筛选或聚合。

In [4]:
# 输入全部明细与最新评价；返回一行一个订单的新表。
order_reviews = aggregate_order_reviews(reviews_clean, latest_reviews)

print('订单级评价表行数：', len(order_reviews))
print('订单ID重复数：', order_reviews['order_id'].duplicated().sum())

# 聚合后行数减少，但通过计数列求和可以追溯所有来源记录。
print('汇总计数列记录的评价数：', order_reviews['review_record_count'].sum())
print('汇总计数列记录的低评分数：', order_reviews['low_score_record_count'].sum())

order_reviews[
    ['order_id', 'review_record_count', 'low_score_record_count',
     'latest_review_score', 'latest_is_low_score', 'any_low_score']
].head()

订单级评价表行数： 98673
订单ID重复数： 0
汇总计数列记录的评价数： 99224
汇总计数列记录的低评分数： 14575


,order_id,review_record_count,low_score_record_count,latest_review_score,latest_is_low_score,any_low_score
0,00010242fe8c5a6d1ba2dd792cb16214,1,0,5,False,False
1,00018f77f2f0320c557190d7a144bdd3,1,0,4,False,False
2,000229ec398224ef6ca0657da4fc703e,1,0,5,False,False
3,00024acbcdf0a6daa1e931b038114c75,1,0,4,False,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,0,5,False,False


In [5]:
# 两种口径都在同一批有有效评价的订单中比较。
display(compare_review_rules(order_reviews))
print('两种规则判定不同的订单数：', order_reviews['rules_disagree'].sum())

# 从真实数据里选择一笔规则不同的订单，追溯完整评价记录。
disagreements = order_reviews.loc[order_reviews['rules_disagree']].sort_values('order_id')
if not disagreements.empty:
    example_order_id = disagreements.iloc[0]['order_id']

    # 明细里应看到先 2 分、后 5 分；两条记录都仍然存在。
    display(reviews_clean.loc[
        reviews_clean['order_id'].eq(example_order_id),
        ['order_id', 'review_id', 'review_score', 'review_answer_timestamp']
    ].sort_values('review_answer_timestamp'))

    # 订单级表只有一行，同时保存最新 5 分与“曾经有低分”的标记。
    display(order_reviews.loc[
        order_reviews['order_id'].eq(example_order_id),
        ['order_id', 'review_record_count', 'low_score_record_count',
         'latest_review_score', 'latest_is_low_score', 'any_low_score']
    ])

,低评分订单数,有有效评价订单数,低评分比例（%）
最新有效评分,14494,98673,14.69
任意一次低评分,14533,98673,14.73


两种规则判定不同的订单数： 39


,order_id,review_id,review_score,review_answer_timestamp
15775,059bbeb3477ed66fd7e670c3f879009a,47746f14e9ac9954d196de1cfb45741e,2,2017-08-01 20:39:04
9139,059bbeb3477ed66fd7e670c3f879009a,85b0c36f463e19aa709b2ee1446513ce,5,2017-08-04 09:32:59


,order_id,review_record_count,low_score_record_count,latest_review_score,latest_is_low_score,any_low_score
2189,059bbeb3477ed66fd7e670c3f879009a,2,1,5,False,True


## 4. 关联回所有订单，保留无评价的未知状态

join_orders_reviews 先反向检查评价订单在主表中是否存在，再做一对一左连接。订单表全部保留。

没有评价时，latest_review_score、latest_is_low_score、any_low_score 都保持未知。False 的意思是有有效评价且规则判定为非低分。

reconcile_review_totals 检查来源记录数量、低评分记录数量、有评价订单数和最新评价订单数在各层是否一致。

In [6]:
# 注意输入是订单原表和订单级评价，不是直接把全部评价明细连接到主表。
orders_with_reviews = join_orders_reviews(raw_orders, order_reviews)

# 差额列应全部为 0；行数不同通过汇总计数列核对，不要求所有表长度相同。
review_controls = reconcile_review_totals(
    reviews_clean, latest_reviews, order_reviews, orders_with_reviews
)
display(review_controls)

print('关联后主表订单数：', len(orders_with_reviews))
display(orders_with_reviews['reviews_match'].value_counts())

# 无评价订单仍保留，评分与低评分标记为 <NA>。
missing_review_orders = orders_with_reviews.loc[
    orders_with_reviews['flag_missing_reviews']
].copy()
display(missing_review_orders['order_status'].value_counts())
missing_review_orders[
    ['order_id', 'order_status', 'latest_review_score',
     'latest_is_low_score', 'any_low_score']
].head()

,明细层,订单汇总层,关联后,汇总差额,关联差额
评价记录数,99224,99224,99224,0,0
低评分评价记录数,14575,14575,14575,0,0
有评价订单数,98673,98673,98673,0,0
最新评价对应订单数,98673,98673,98673,0,0


关联后主表订单数： 99441


reviews_match
both          98673
left_only       768
right_only        0
Name: count, dtype: int64

order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64

,order_id,order_status,latest_review_score,latest_is_low_score,any_low_score
16,403b97836b0c04a622354cf531062e5f,delivered,<NA>,<NA>,<NA>
154,6942b8da583c2f9957e990d028607019,shipped,<NA>,<NA>,<NA>
311,4906eeadde5f70b308c20c4a8f20be02,delivered,<NA>,<NA>,<NA>
382,b7a4a9ecb1cd3ef6a3e36a48e200e3be,delivered,<NA>,<NA>,<NA>
390,59b32faedc12322c672e95ec3716d614,delivered,<NA>,<NA>,<NA>


In [7]:
# 指标字典可供终端导出脚本保存为 JSON，下面只挑主要基线查看。
metrics = summarize_review_metrics(
    reviews_clean, latest_reviews, order_reviews, orders_with_reviews
)
display(pd.Series({
    '评价明细行数': metrics['review_rows'],
    '有有效评价订单数': metrics['reviewed_orders'],
    '低评分评价记录数': metrics['low_score_review_rows'],
    '任意一次低分订单数': metrics['any_low_score_orders'],
    '最新一次低分订单数': metrics['latest_low_score_orders'],
    '规则不同订单数': metrics['rules_disagree_orders'],
    '无评价订单数': metrics['orders_without_reviews']
}))

# 长间隔样本保留以便核查；间隔包含等待时间，不能解释为填写问卷的用时。
reviews_clean.nlargest(3, 'review_answer_lag_days')[
    ['order_id', 'review_id', 'review_creation_date',
     'review_answer_timestamp', 'review_answer_lag_days']
]

评价明细行数       99224
有有效评价订单数     98673
低评分评价记录数     14575
任意一次低分订单数    14533
最新一次低分订单数    14494
规则不同订单数         39
无评价订单数         768
dtype: int64

,order_id,review_id,review_creation_date,review_answer_timestamp,review_answer_lag_days
41620,bb5849f8ba21da43ffa31ea52ba81b37,40dad6438b6cbec46d936bec2377778c,2017-03-24,2018-08-24 16:46:52,518.699213
66672,a8a2941e92152e0edc39bb862cd6b6e2,eeb94b15f672afaf38e8d74b6ddb25f8,2017-02-14,2018-07-11 15:47:37,512.658067
22059,898ded7ecb1e4d92997856bd8b7a8aa1,083e71bf744a3219a07a948f12243f9a,2017-03-28,2018-08-18 01:36:09,508.066771


## 导出和下一步

本 Notebook 只读取原始文件并在内存中复现。项目目录下运行 python .\src\05_clean_reviews.py，将生成 data/processed/reviews_v1 和 reports/reviews_v1 下的版本化结果、清单与阶段报告。

当前是全量历史、全部状态的课堂基线。最新评分作为主口径，任意一次低评分作为对照；正式分析窗口与历史时点评分可见性留到后续明确。

你的第八份 Notebook 保留完整手写过程。本次复现完成后，下一课进入客户表的主键、客户身份、地区与邮编检查。

## 我的理解

为什么 reviews_clean 有 99,224 行，而 order_reviews 只有 98,673 行，控制总额表仍然可以全部通过？具体用 order_reviews 中哪一列核对来源评价记录数？

我的回答：两张表的粒度不同：`reviews_clean` 一行一条评价，`order_reviews` 一行一个订单的评价汇总。汇总表通过 review_record_count 保留每单的来源评价数量，该列求和仍为 99,224，因此数量核对可以通过。

order_reviews 同时保存“最新一条评价的字段”和“全部评价的汇总信息”
